In [1]:
from __future__ import print_function, division
import os, random, time, copy
import numpy as np
import pandas as pd
import os.path as path
import scipy.io as sio
from scipy import misc
from scipy import ndimage, signal
import scipy
import pickle
import sys
import math
import matplotlib.pyplot as plt
from io import BytesIO


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision
from torchvision import datasets, models, transforms
import torchvision.utils as vutils


import warnings 
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)


manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)


import faiss
import os
import pandas as pd
import seaborn as sn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import faiss
from IPython import display
from torch.optim.lr_scheduler import OneCycleLR


3.9.18 (main, Sep 11 2023, 14:09:26) [MSC v.1916 64 bit (AMD64)]
1.12.1
Random Seed:  999


In [2]:

tarad=torch.load('./savedata/tarad1225.pkl')



featesttrain=torch.load('./savedata/featesttrain1229.pkl')

featestA=torch.load('./savedata/featestA1229.pkl')

featestD=torch.load('./savedata/featestD1229.pkl')



featesttrain2=torch.load('./savedata/featesttrain0102.pkl')

featestA2=torch.load('./savedata/featestA0102.pkl')

featestD2=torch.load('./savedata/featestD0102.pkl')




In [3]:
dd=[]
newtrain=[]
for i in range(len(featesttrain)):
    dd=[]
    dd.append(featesttrain[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtrain.append(dd)
    
    
dd=[]
newtestA=[]
for i in range(len(featestA)):
    dd=[]
    dd.append(featestA[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtestA.append(dd)
    
dd=[]
newtestD=[]
for i in range(len(featestD)):
    dd=[]
    dd.append(featestD[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtestD.append(dd)

    
train_features = np.array(torch.tensor(newtrain))
test_features = np.array(torch.tensor(newtestA))
ood_features = np.array(torch.tensor(newtestD))

In [4]:
dd=[]
newtrain2=[]
for i in range(len(featesttrain2)):
    dd=[]
    dd.append(featesttrain2[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtrain2.append(dd)
    
    
dd=[]
newtestA2=[]
for i in range(len(featestA2)):
    dd=[]
    dd.append(featestA2[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtestA2.append(dd)
    
dd=[]
newtestD2=[]
for i in range(len(featestD2)):
    dd=[]
    dd.append(featestD2[i].detach().cpu().numpy())
    dd=np.concatenate(dd)
    newtestD2.append(dd)
  

train_features2 = np.array(torch.tensor(newtrain2))
test_features2 = np.array(torch.tensor(newtestA2))
ood_features2 = np.array(torch.tensor(newtestD2))

In [5]:
min_train_size = min(train_features.shape[0], train_features2.shape[0])
min_test_size = min(test_features.shape[0], test_features2.shape[0])
min_ood_size = min(ood_features.shape[0], ood_features2.shape[0])

train_features = train_features[:min_train_size]
train_features2 = train_features2[:min_train_size]

test_features = test_features[:min_test_size]
test_features2 = test_features2[:min_test_size]

ood_features = ood_features[:min_ood_size]
ood_features2 = ood_features2[:min_ood_size]

train_features00 = np.concatenate([train_features, train_features2], axis=1)
test_features00 = np.concatenate([test_features, test_features2], axis=1)
ood_features00 = np.concatenate([ood_features, ood_features2], axis=1)


In [6]:
index = faiss.IndexFlatL2(train_features00.shape[1])
index.add(train_features00)

In [7]:
k = 1
in_l2_distances, in_k_closest_points = index.search(test_features00, k)

In [8]:
%%time

k = 1
in_l2_distances, in_k_closest_points = index.search(test_features00, k)
in_scores = in_l2_distances[:, -1]

CPU times: total: 0 ns
Wall time: 0 ns


In [27]:

out_l2_distances, out_k_closest_points = index.search(ood_features00, k)

In [28]:

out_scores = out_l2_distances[:, -1]

In [29]:
in_scores.sort()
out_scores.sort()

In [30]:

threshold =17.89636 #Thresholds can be selected from the validation set

In [31]:
print("threshold:",threshold)

tp = 0
tn = 0
fp = 0
fn = 0
testAres=[]
testDres=[]
for point in in_scores:
    if point <= threshold:
        testAres.append(0)
        tp += 1
    else:
        testAres.append(1)
        fn += 1

for point in out_scores:
    if point <= threshold:
        testDres.append(0)
        fp += 1
    else:
        testDres.append(1)
        tn += 1
        
tp_rate = tp / (tp + fn)
tn_rate = tn / (tn + fp)
fp_rate = fp / (fp + tn)
fn_rate = fn / (fn + tp)

result = {
    "tp": tp,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tpr": tp_rate,
    "fpr": fp_rate,
}

print(result)

threshold: 17.89636
{'tp': 147100, 'tn': 10752, 'fp': 0, 'fn': 100, 'tpr': 0.9993206521739131, 'fpr': 0.0}


In [32]:
testres=testAres+testDres

In [33]:
y_test=[]

for i in range(len(tarad)):
    if tarad[i]<3:
        y_test.append(0)
    else:
        y_test.append(1)

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 假设 y_test 是真实标签，testres 是预测结果
accuracy = accuracy_score(y_test, testres)
precision = precision_score(y_test, testres, average='weighted')  # 或 'macro', 'micro' 取决于你的需求
recall = recall_score(y_test, testres, average='weighted')  # 或 'macro', 'micro'
f1 = f1_score(y_test, testres, average='weighted')  # 或 'macro', 'micro'

print("Accuracy of DeepkNN in localizing unknown attacks: ", accuracy)
print("Precision of DeepkNN in localizing unknown attacks: ", precision)
print("Recall of DeepkNN in localizing unknown attacks: ", recall)
print("F1 Score of DeepkNN in localizing unknown attacks: ", f1)
conf_matrix = confusion_matrix(y_test, testres)
print("Confusion Matrix:")
print(conf_matrix)
num_classes = conf_matrix.shape[0]

# 初始化结果存储
accuracies = []
precisions = []
recalls = []
f1s = []

for i in range(num_classes):
    TP = conf_matrix[i, i]
    FP = conf_matrix[:, i].sum() - TP
    FN = conf_matrix[i, :].sum() - TP
    TN = conf_matrix.sum() - (TP + FP + FN)

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

# 打印结果
for i in range(num_classes):
    print(f"Class {i}:")
    print(f"  Accuracy: {accuracies[i]:.4f}")
    print(f"  Precision: {precisions[i]:.4f}")
    print(f"  Recall: {recalls[i]:.4f}")
    print(f"  F1 Score: {f1s[i]:.4f}")


Accuracy of DeepkNN in localizing unknown attacks:  0.9993668962722853
Precision of DeepkNN in localizing unknown attacks:  0.9993727302542952
Recall of DeepkNN in localizing unknown attacks:  0.9993668962722853
F1 Score of DeepkNN in localizing unknown attacks:  0.9993682539578856
Confusion Matrix:
[[147100    100]
 [     0  10752]]
Class 0:
  Accuracy: 0.9994
  Precision: 1.0000
  Recall: 0.9993
  F1 Score: 0.9997
Class 1:
  Accuracy: 0.9994
  Precision: 0.9908
  Recall: 1.0000
  F1 Score: 0.9954


In [169]:
torch.save(testres,'./savedata/testres1231.pkl')

In [10]:
print(train_features.shape)
print(train_features2.shape)
print(test_features.shape)
print(test_features2.shape)
print(ood_features.shape)
print(ood_features2.shape)


(588672, 32)
(588720, 64)
(147200, 32)
(147248, 64)
(10752, 32)
(10752, 64)


In [4]:
newtestD

[array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.2110375 , 0.        , 0.        , 0.21091919, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.37986878, 0.25093865, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.8388718 , 0.        , 0.        ,
        0.        , 0.        ], dtype=float32),
 array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.2110375 , 0.        , 0.        , 0.21091919, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.37986878, 0.25093865, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.8388718 , 0.        , 0.        ,
        0.        , 0.        ], dtype=float32),
 array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.2110375 